###  <font color="#e6e607"> Logistic Dataset Creation </font>

In [1]:
import pandas as pd
import numpy as np

data = {
    "Shipment_ID": [101, 102, 103, 104, 105, 106, 107, 108, 108, 110],
    "Distance_KM": [10, 25, 15, np.nan, 40, 30, 20, 35, 35, 500],
    "Delivery_Time_Hours": [2, 5, 3, 4, np.nan, 6, 3, 7, 7, 48],
    "Shipping_Cost": [100, 250, 150, 200, 400, np.nan, 180, 350, 350, 5000],
    "Delivery_Status": [
        "On Time", "Delayed", "On Time", "On Time", "Delayed",
        "On Time", "On Time", "Delayed", "Delayed", "Delayed"
    ]
}

df = pd.DataFrame(data)

df.to_csv("logistics_data.csv", index=False)

print("Dataset created successfully!")
print(df)

Dataset created successfully!
   Shipment_ID  Distance_KM  Delivery_Time_Hours  Shipping_Cost  \
0          101         10.0                  2.0          100.0   
1          102         25.0                  5.0          250.0   
2          103         15.0                  3.0          150.0   
3          104          NaN                  4.0          200.0   
4          105         40.0                  NaN          400.0   
5          106         30.0                  6.0            NaN   
6          107         20.0                  3.0          180.0   
7          108         35.0                  7.0          350.0   
8          108         35.0                  7.0          350.0   
9          110        500.0                 48.0         5000.0   

  Delivery_Status  
0         On Time  
1         Delayed  
2         On Time  
3         On Time  
4         Delayed  
5         On Time  
6         On Time  
7         Delayed  
8         Delayed  
9         Delayed  


 ###  <font color="#e6e607"> Data-quality inspection </font>

In [2]:

print("Dataset Shape:")
print(df.shape)

print("\nColumn Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nStatistical Summary:")
print(df.describe())


Dataset Shape:
(10, 5)

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Shipment_ID          10 non-null     int64  
 1   Distance_KM          9 non-null      float64
 2   Delivery_Time_Hours  9 non-null      float64
 3   Shipping_Cost        9 non-null      float64
 4   Delivery_Status      10 non-null     str    
dtypes: float64(3), int64(1), str(1)
memory usage: 602.0 bytes

Missing Values:
Shipment_ID            0
Distance_KM            1
Delivery_Time_Hours    1
Shipping_Cost          1
Delivery_Status        0
dtype: int64

Duplicate Rows:
1

Statistical Summary:
       Shipment_ID  Distance_KM  Delivery_Time_Hours  Shipping_Cost
count     10.00000     9.000000             9.000000       9.000000
mean     105.40000    78.888889             9.444444     775.555556
std        2.91357   158.228035            14.569756 

###  <font color="#e6e607"> Create a working copy of the original dataset </font> 

In [4]:
df_clean = df.copy()

# <font color="#e6e607">Handle Missing Values </font>
###  <font color="#e6e607">Fill missing numerical values using the median</font>


In [5]:
numeric_cols = [
    "Distance_KM",
    "Delivery_Time_Hours",
    "Shipping_Cost"
]

for col in numeric_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)

print("Missing values handled successfully!")
print(df_clean.isnull().sum())


Missing values handled successfully!
Shipment_ID            0
Distance_KM            0
Delivery_Time_Hours    0
Shipping_Cost          0
Delivery_Status        0
dtype: int64


#  <font color="#e6e607"> Remove Duplicate Rows </font>

In [7]:

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates()

rows_after = len(df_clean)

print("Rows before cleaning:", rows_before)
print("Rows after removing exact duplicates:", rows_after)
print("Duplicate rows remaining:", df_clean.duplicated().sum())

Rows before cleaning: 9
Rows after removing exact duplicates: 9
Duplicate rows remaining: 0


###  <font color="#e6e607"> Verify data Cleaning </font>

In [8]:

print(df_clean.head(10))
print("\nFinal dataset shape:", df_clean.shape)
print("\nMissing values:\n", df_clean.isnull().sum())


   Shipment_ID  Distance_KM  Delivery_Time_Hours  Shipping_Cost  \
0          101         10.0                  2.0          100.0   
1          102         25.0                  5.0          250.0   
2          103         15.0                  3.0          150.0   
3          104         30.0                  4.0          200.0   
4          105         40.0                  5.0          400.0   
5          106         30.0                  6.0          250.0   
6          107         20.0                  3.0          180.0   
7          108         35.0                  7.0          350.0   
9          110        500.0                 48.0         5000.0   

  Delivery_Status  
0         On Time  
1         Delayed  
2         On Time  
3         On Time  
4         Delayed  
5         On Time  
6         On Time  
7         Delayed  
9         Delayed  

Final dataset shape: (9, 5)

Missing values:
 Shipment_ID            0
Distance_KM            0
Delivery_Time_Hours    0
Shippin

###  <font color="#e6e607"> Outliers Detecting </font>

In [9]:

outlier_cols = [
    "Distance_KM",
    "Delivery_Time_Hours",
    "Shipping_Cost"
]

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    ]

    print(f"\nColumn: {col}")
    print("Lower bound:", round(lower_bound, 2))
    print("Upper bound:", round(upper_bound, 2))
    print("Outlier count:", len(outliers))
    print("Outlier values:", outliers[col].tolist())



Column: Distance_KM
Lower bound: -2.5
Upper bound: 57.5
Outlier count: 1
Outlier values: [500.0]

Column: Delivery_Time_Hours
Lower bound: -1.5
Upper bound: 10.5
Outlier count: 1
Outlier values: [48.0]

Column: Shipping_Cost
Lower bound: -75.0
Upper bound: 605.0
Outlier count: 1
Outlier values: [5000.0]


###  <font color="#e6e607"> Handle Outlires </font> 

In [10]:
df_processed = df_clean.copy()

for col in outlier_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_processed[col] = df_processed[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Outlier capping completed.")
print(df_processed[outlier_cols].describe())


Outlier capping completed.
       Distance_KM  Delivery_Time_Hours  Shipping_Cost
count     9.000000             9.000000       9.000000
mean     29.166667             5.055556     276.111111
std      14.252193             2.579298     155.035838
min      10.000000             2.000000     100.000000
25%      20.000000             3.000000     180.000000
50%      30.000000             5.000000     250.000000
75%      35.000000             6.000000     350.000000
max      57.500000            10.500000     605.000000


###  <font color="#e6e607">  Before Outlires and After Handle Outlires Comparision </font>

In [11]:
for col in outlier_cols:
    print(f"\n{col}")
    print("Before:", df_clean[col].max())
    print("After :", df_processed[col].max())


Distance_KM
Before: 500.0
After : 57.5

Delivery_Time_Hours
Before: 48.0
After : 10.5

Shipping_Cost
Before: 5000.0
After : 605.0


###  <font color="#e6e607"> Min Max Normalization </font>

In [12]:
df_normalized = df_processed.copy()

for col in outlier_cols:
    min_val = df_normalized[col].min()
    max_val = df_normalized[col].max()

    if max_val > min_val:
        df_normalized[col + "_MinMax"] = (
            (df_normalized[col] - min_val) /
            (max_val - min_val)
        )
    else:
        df_normalized[col + "_MinMax"] = 0.0

print("Min-Max normalization completed!")
print(df_normalized.head())

Min-Max normalization completed!
   Shipment_ID  Distance_KM  Delivery_Time_Hours  Shipping_Cost  \
0          101         10.0                  2.0          100.0   
1          102         25.0                  5.0          250.0   
2          103         15.0                  3.0          150.0   
3          104         30.0                  4.0          200.0   
4          105         40.0                  5.0          400.0   

  Delivery_Status  Distance_KM_MinMax  Delivery_Time_Hours_MinMax  \
0         On Time            0.000000                    0.000000   
1         Delayed            0.315789                    0.352941   
2         On Time            0.105263                    0.117647   
3         On Time            0.421053                    0.235294   
4         Delayed            0.631579                    0.352941   

   Shipping_Cost_MinMax  
0              0.000000  
1              0.297030  
2              0.099010  
3              0.198020  
4              0.59

###  <font color="#e6e607"> Z-Score Standardization </font> 

In [13]:
for col in outlier_cols:
    mean_val = df_normalized[col].mean()
    std_val = df_normalized[col].std()

    if pd.notna(std_val) and std_val > 0:
        df_normalized[col + "_ZScore"] = (
            (df_normalized[col] - mean_val) / std_val
        )
    else:
        df_normalized[col + "_ZScore"] = 0.0

print("Z-score standardization completed!")
print(df_normalized.head())

Z-score standardization completed!
   Shipment_ID  Distance_KM  Delivery_Time_Hours  Shipping_Cost  \
0          101         10.0                  2.0          100.0   
1          102         25.0                  5.0          250.0   
2          103         15.0                  3.0          150.0   
3          104         30.0                  4.0          200.0   
4          105         40.0                  5.0          400.0   

  Delivery_Status  Distance_KM_MinMax  Delivery_Time_Hours_MinMax  \
0         On Time            0.000000                    0.000000   
1         Delayed            0.315789                    0.352941   
2         On Time            0.105263                    0.117647   
3         On Time            0.421053                    0.235294   
4         Delayed            0.631579                    0.352941   

   Shipping_Cost_MinMax  Distance_KM_ZScore  Delivery_Time_Hours_ZScore  \
0              0.000000           -1.344822                   -1.184646 

###  <font color="#e6e607"> Verify Result </font> 

In [14]:
normalized_cols = [
    col + "_MinMax" for col in outlier_cols
]

standardized_cols = [
    col + "_ZScore" for col in outlier_cols
]

print("Min-Max normalized ranges:")
print(df_normalized[normalized_cols].agg(["min", "max"]))

print("\nZ-score standardized means:")
print(df_normalized[standardized_cols].mean().round(4))

Min-Max normalized ranges:
     Distance_KM_MinMax  Delivery_Time_Hours_MinMax  Shipping_Cost_MinMax
min                 0.0                         0.0                   0.0
max                 1.0                         1.0                   1.0

Z-score standardized means:
Distance_KM_ZScore           -0.0
Delivery_Time_Hours_ZScore    0.0
Shipping_Cost_ZScore          0.0
dtype: float64


###  <font color="#e6e607"> Cleaned Dataset Export </font>

In [15]:
df_normalized.to_csv(
    "logistics_preprocessed.csv",
    index=False
)

print("Final preprocessed dataset exported successfully!")
print("Final dataset shape:", df_normalized.shape)

Final preprocessed dataset exported successfully!
Final dataset shape: (9, 11)


# <font color="#e6e607"> Final Validation </font>

In [16]:
print("Final dataset shape:", df_normalized.shape)

print("\nMissing values:")
print(df_normalized.isnull().sum())

print("\nDuplicate rows:", df_normalized.duplicated().sum())

print("\nData types:")
print(df_normalized.dtypes)

print("\nMin-Max normalized columns:")
print(df_normalized[normalized_cols].agg(["min", "max"]))

print("\nStandardized column means:")
print(df_normalized[standardized_cols].mean().round(4))

print("\nFirst 5 processed records:")
print(df_normalized.head())

# Confirm that the CSV was exported
import os
print("\nCSV file exists:", os.path.exists("logistics_preprocessed.csv"))

Final dataset shape: (9, 11)

Missing values:
Shipment_ID                   0
Distance_KM                   0
Delivery_Time_Hours           0
Shipping_Cost                 0
Delivery_Status               0
Distance_KM_MinMax            0
Delivery_Time_Hours_MinMax    0
Shipping_Cost_MinMax          0
Distance_KM_ZScore            0
Delivery_Time_Hours_ZScore    0
Shipping_Cost_ZScore          0
dtype: int64

Duplicate rows: 0

Data types:
Shipment_ID                     int64
Distance_KM                   float64
Delivery_Time_Hours           float64
Shipping_Cost                 float64
Delivery_Status                   str
Distance_KM_MinMax            float64
Delivery_Time_Hours_MinMax    float64
Shipping_Cost_MinMax          float64
Distance_KM_ZScore            float64
Delivery_Time_Hours_ZScore    float64
Shipping_Cost_ZScore          float64
dtype: object

Min-Max normalized columns:
     Distance_KM_MinMax  Delivery_Time_Hours_MinMax  Shipping_Cost_MinMax
min                 0.